In [22]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

INTEGRATED_PATH = "../data/integrated/algeria_wildfire_dataset.parquet"
OUT_PATH        = "../data/integrated/algeria_wildfire_dataset_features.parquet"

In [23]:
df = pd.read_parquet(INTEGRATED_PATH)

df = df.sort_values(["wilaya_id", "date"]).reset_index(drop=True)

# Fix soil_moisture tiny negatives from ERA5 numerical noise
df["soil_moisture"] = df["soil_moisture"].clip(lower=0.0)

print(f"Loaded: {df.shape}")
print(f"soil_moisture negatives after clip: {(df['soil_moisture'] < 0).sum()}")

Loaded: (95664, 32)
soil_moisture negatives after clip: 0


In [24]:
g = df.groupby("wilaya_id")

# Temperature rolling means
df["temp_mean_3d"]  = g["temp_c"].transform(lambda x: x.rolling(3,  min_periods=1).mean())
df["temp_mean_7d"]  = g["temp_c"].transform(lambda x: x.rolling(7,  min_periods=1).mean())
df["temp_mean_14d"] = g["temp_c"].transform(lambda x: x.rolling(14, min_periods=1).mean())

# RH rolling means
df["rh_mean_7d"]    = g["rh"].transform(lambda x: x.rolling(7,  min_periods=1).mean())
df["rh_min_7d"]     = g["rh"].transform(lambda x: x.rolling(7,  min_periods=1).min())

# Precipitation rolling sums
df["rain_3d"]       = g["precip_mm"].transform(lambda x: x.rolling(3,  min_periods=1).sum())
df["rain_7d"]       = g["precip_mm"].transform(lambda x: x.rolling(7,  min_periods=1).sum())
df["rain_14d"]      = g["precip_mm"].transform(lambda x: x.rolling(14, min_periods=1).sum())
df["rain_30d"]      = g["precip_mm"].transform(lambda x: x.rolling(30, min_periods=1).sum())

print(f"  Shape: {df.shape}")

  Shape: (95664, 41)


In [25]:
def days_since_rain(series):
    """Count days since last precipitation > 1mm."""
    values = series.values
    result = np.zeros(len(values))
    count  = 0
    for i, val in enumerate(values):
        count  = 0 if val > 1.0 else count + 1
        result[i] = count
    return pd.Series(result, index=series.index)

def consecutive_hot_days(series, threshold=35.0):
    """Count consecutive days above temperature threshold."""
    values = series.values
    result = np.zeros(len(values))
    count  = 0
    for i, val in enumerate(values):
        count  = count + 1 if val > threshold else 0
        result[i] = count
    return pd.Series(result, index=series.index)

df["days_since_rain"]  = g["precip_mm"].transform(days_since_rain)
df["hot_days_streak"]  = g["temp_c"].transform(consecutive_hot_days)

print(f"  Max days_since_rain : {df['days_since_rain'].max():.0f}")
print(f"  Max hot_days_streak : {df['hot_days_streak'].max():.0f}")

  Max days_since_rain : 986
  Max hot_days_streak : 154


In [26]:
df["FWI_mean_7d"]  = g["FWI"].transform(lambda x: x.rolling(7,  min_periods=1).mean())
df["FWI_mean_14d"] = g["FWI"].transform(lambda x: x.rolling(14, min_periods=1).mean())
df["FWI_max_7d"]   = g["FWI"].transform(lambda x: x.rolling(7,  min_periods=1).max())
df["FWI_max_14d"]  = g["FWI"].transform(lambda x: x.rolling(14, min_periods=1).max())
df["FWI_trend"]    = g["FWI"].transform(lambda x: x.diff().fillna(0))
df["DC_14d_mean"]  = g["DC"].transform(lambda x: x.rolling(14, min_periods=1).mean())

In [27]:
# fires_last_7d and fires_last_30d REMOVED — cause data leakage
# (current-season fire counts directly encode the label)
# fires_prev_year_month kept — uses last year's data, no leakage

prev_year = (df.groupby(["wilaya_id", "year", "month"])["fire_count"]
               .sum()
               .reset_index()
               .rename(columns={"fire_count": "fires_prev_year_month",
                                 "year": "prev_year"}))
prev_year["year"] = prev_year["prev_year"] + 1
df = df.merge(
    prev_year[["wilaya_id", "year", "month", "fires_prev_year_month"]],
    on=["wilaya_id", "year", "month"],
    how="left"
)
df["fires_prev_year_month"] = df["fires_prev_year_month"].fillna(0)

print(f"  fires_prev_year_month — mean: {df['fires_prev_year_month'].mean():.2f}  max: {df['fires_prev_year_month'].max():.0f}")
print("  ✓ fires_last_7d and fires_last_30d excluded (leakage)")

  fires_prev_year_month — mean: 81.35  max: 5136
  ✓ fires_last_7d and fires_last_30d excluded (leakage)


In [28]:
# NDVI month-over-month change per wilaya (vegetation drying signal)
df["NDVI_change"] = g["NDVI"].transform(lambda x: x.diff().fillna(0))
df["NBR_change"]  = g["NBR"].transform(lambda x: x.diff().fillna(0))


In [29]:
# Vegetation × fire danger
df["NDVI_FWI"]          = df["NDVI"] * df["FWI"]
df["NBR_FWI"]           = df["NBR"]  * df["FWI"]

# Human ignition pressure
df["ignition_pressure"] = df["pop_density_mean"] / (df["road_distance_mean_km"] + 1)

# Slope × wind (terrain-driven spread potential)
df["slope_wind"]        = df["slope_mean_deg"] * df["wind_speed_kmh"]

# Soil dryness × heat
df["heat_drought"]      = df["temp_c"] * (1 - df["soil_moisture"])

In [30]:
df["month_sin"]     = np.sin(2 * np.pi * df["month"] / 12)
df["month_cos"]     = np.cos(2 * np.pi * df["month"] / 12)
df["doy_sin"]       = np.sin(2 * np.pi * df["day_of_year"] / 365)
df["doy_cos"]       = np.cos(2 * np.pi * df["day_of_year"] / 365)

In [35]:
new_features = [
    "temp_mean_3d", "temp_mean_7d", "temp_mean_14d",
    "rh_mean_7d", "rh_min_7d",
    "rain_3d", "rain_7d", "rain_14d", "rain_30d",
    "days_since_rain", "hot_days_streak",
    "FWI_mean_7d", "FWI_mean_14d", "FWI_max_7d", "FWI_max_14d",
    "FWI_trend", "DC_14d_mean",
    "fires_prev_year_month",
    "NDVI_change", "NBR_change",
    "NDVI_FWI", "NBR_FWI", "ignition_pressure",
    "slope_wind", "heat_drought",
    "month_sin", "month_cos", "doy_sin", "doy_cos",
]

print(f"New features added : {len(new_features)}")
print(f"Total columns      : {df.shape[1]}")
print(f"Total rows         : {df.shape[0]:,}")

print("\nMissing values in new features:")
missing = df[new_features].isnull().sum()
missing = missing[missing > 0]
print(missing if len(missing) > 0 else "  ✓ None")

print("\nInfinite values in new features:")
inf_count = np.isinf(df[new_features].select_dtypes(include=np.number)).sum().sum()
print(f"  {'✓ None' if inf_count == 0 else f'⚠ {inf_count} found'}")

print("\nSample stats on key new features:")
check = ["days_since_rain", "hot_days_streak", "FWI_mean_7d",
            "fires_prev_year_month", "ignition_pressure"]
print(df[check].describe().round(2).to_string())

New features added : 29
Total columns      : 61
Total rows         : 95,664

Missing values in new features:
  ✓ None

Infinite values in new features:
  ✓ None

Sample stats on key new features:
       days_since_rain  hot_days_streak  FWI_mean_7d  fires_prev_year_month  ignition_pressure
count         95664.00         95664.00     95664.00               95664.00           95664.00
mean             36.80             3.55        39.33                  81.35             187.06
std              84.81            13.11        24.19                 280.84             663.91
min               0.00             0.00         0.01                   0.00               0.01
25%               2.00             0.00        21.49                   2.00               7.28
50%              10.00             0.00        35.93                  13.00              58.93
75%              32.00             0.00        52.08                  45.00             134.56
max             986.00           154.00     

In [36]:
# ── CELL 11: Quick MI check on new features ───────────────────────
from sklearn.feature_selection import mutual_info_classif

all_features = [
    # Original
    "temp_c", "rh", "precip_mm", "soil_moisture",
    "FFMC", "DMC", "DC", "ISI", "BUI", "FWI",
    "NDVI", "NDWI", "NBR",
    "elevation_mean_m", "slope_mean_deg",
    "pop_density_mean", "road_distance_mean_km",
    "day_of_year",
    # Engineered
] + new_features

all_features = [f for f in all_features if f in df.columns]

X_mi = df[all_features].fillna(0)
y_mi = df["fire_risk_class"]

mi = mutual_info_classif(X_mi, y_mi, random_state=42)
mi_series = pd.Series(mi, index=all_features).sort_values(ascending=False)

print("Top 20 features by Mutual Information (after engineering):")
print(mi_series.head(20).round(4).to_string())
print(f"\nBottom 5 (candidates to drop):")
print(mi_series.tail(5).round(4).to_string())

Top 20 features by Mutual Information (after engineering):
FWI_max_14d              0.2765
FWI_max_7d               0.2432
NBR                      0.1955
NDVI                     0.1920
fires_prev_year_month    0.1624
pop_density_mean         0.1575
ignition_pressure        0.1567
slope_mean_deg           0.1548
elevation_mean_m         0.1539
road_distance_mean_km    0.1529
NDWI                     0.1484
rain_30d                 0.0990
rh_min_7d                0.0846
rain_14d                 0.0695
soil_moisture            0.0438
rain_7d                  0.0409
heat_drought             0.0384
NBR_FWI                  0.0384
temp_c                   0.0363
FWI                      0.0348

Bottom 5 (candidates to drop):
doy_cos        0.0055
month_cos      0.0052
FWI_trend      0.0047
NDVI_change    0.0010
NBR_change     0.0000


In [37]:
FEATURES = [
    # Core weather
    "temp_c", "rh", "precip_mm", "soil_moisture",

    # Rolling weather
    "temp_mean_7d",
    "rh_mean_7d", "rh_min_7d",
    "rain_7d", "rain_14d", "rain_30d",
    "days_since_rain", "hot_days_streak",

    # FWI system
    "FFMC", "DMC", "DC", "ISI", "BUI", "FWI",

    # Rolling FWI
    "FWI_mean_7d", "FWI_max_7d", "FWI_max_14d",

    # Vegetation
    "NDVI", "NDWI", "NBR",

    # Terrain
    "elevation_mean_m", "slope_mean_deg",

    # Human geography
    "pop_density_mean", "road_distance_mean_km", "ignition_pressure",

    # Fire history (last year only — no leakage)
    "fires_prev_year_month",

    # Temporal
    "day_of_year", "month_sin",

    # Interactions
    "NBR_FWI", "heat_drought",
]

DROPPED = {
    "fires_last_7d":  "DATA LEAKAGE — current season fire counts encode the label",
    "fires_last_30d": "DATA LEAKAGE — current season fire counts encode the label",
    "FWI_trend":      "MI=0.005 — noise",
    "month_cos":      "MI=0.004 — redundant with month_sin",
    "doy_sin":        "MI=0.003 — redundant with day_of_year",
    "doy_cos":        "MI=0.003 — noise",
    "NDVI_change":    "MI=0.000 — zero signal",
    "NBR_change":     "MI=0.000 — zero signal",
    "temp_mean_3d":   "redundant — temp_mean_7d covers it",
    "temp_mean_14d":  "redundant — temp_mean_7d sufficient",
    "FWI_mean_14d":   "redundant — FWI_max_14d dominates",
    "DC_14d_mean":    "redundant — DC already in features",
    "rain_3d":        "redundant — rain_7d covers it",
    "NDVI_FWI":       "redundant — NBR_FWI is stronger",
    "slope_wind":     "redundant — slope_mean_deg alone sufficient",
}

print(f"Final feature count : {len(FEATURES)}")
print(f"Dropped             : {len(DROPPED)}")
print(f"\nDropped features:")
for feat, reason in DROPPED.items():
    print(f"  ✗ {feat:<24} {reason}")

missing = [f for f in FEATURES if f not in df.columns]
print(f"\n{'✓ All features present in df' if not missing else f'⚠ Missing: {missing}'}")

Final feature count : 34
Dropped             : 15

Dropped features:
  ✗ fires_last_7d            DATA LEAKAGE — current season fire counts encode the label
  ✗ fires_last_30d           DATA LEAKAGE — current season fire counts encode the label
  ✗ FWI_trend                MI=0.005 — noise
  ✗ month_cos                MI=0.004 — redundant with month_sin
  ✗ doy_sin                  MI=0.003 — redundant with day_of_year
  ✗ doy_cos                  MI=0.003 — noise
  ✗ NDVI_change              MI=0.000 — zero signal
  ✗ NBR_change               MI=0.000 — zero signal
  ✗ temp_mean_3d             redundant — temp_mean_7d covers it
  ✗ temp_mean_14d            redundant — temp_mean_7d sufficient
  ✗ FWI_mean_14d             redundant — FWI_max_14d dominates
  ✗ DC_14d_mean              redundant — DC already in features
  ✗ rain_3d                  redundant — rain_7d covers it
  ✗ NDVI_FWI                 redundant — NBR_FWI is stronger
  ✗ slope_wind               redundant — slope_mean

In [38]:
# ── CELL 13: Save final dataset ───────────────────────────────────
# Keep only needed columns to keep file size small
KEEP_COLS = (
    ["date", "wilaya_id", "wilaya_name", "year", "month",
     "fire_risk_class", "fire_risk_label", "fire_count"]
    + FEATURES
)
KEEP_COLS = [c for c in KEEP_COLS if c in df.columns]

df_final = df[KEEP_COLS].copy()
df_final.to_parquet(OUT_PATH, index=False)

print(f"✓ Saved → {OUT_PATH}")
print(f"  Shape    : {df_final.shape}")
print(f"  Features : {len(FEATURES)}")
print(f"  Size     : {df_final.memory_usage(deep=True).sum() / 1e6:.1f} MB")
print(f"\n→ Proceed to 04_model_comparison.ipynb")

✓ Saved → ../data/integrated/algeria_wildfire_dataset_features.parquet
  Shape    : (95664, 42)
  Features : 34
  Size     : 32.4 MB

→ Proceed to 04_model_comparison.ipynb
